In [1]:
import numpy as np
import os
from utils import seed_everything, load_v3_datasets

seed_everything(42)
datasets_v3 = load_v3_datasets("Gd_fps_v3")

# all datasets models comparison

In [8]:
from sklearn.model_selection import KFold
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from collections import defaultdict
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# need tabpfn version 2.5, available at https://huggingface.co/Prior-Labs/tabpfn_2_5
os.environ['HF_TOKEN'] = 'your_token_here'

for folder_name, datasets_dict in datasets_v3.items():
    best_in_folder = defaultdict(int)
    for name, df in datasets_dict.items():
        feature_cols = [c for c in df.columns if c != "lgK"]
        X = df[feature_cols].astype(np.float64).values
        y = df["lgK"].values

        vt = VarianceThreshold(threshold=0)
        X = vt.fit_transform(X)
        if X.shape[1] > 2000:
            vt = VarianceThreshold(threshold=0.01)
            X = vt.fit_transform(X)

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        results = []
        best_rmse, best_model = None, None

        print(f'{folder_name} {name} ({X.shape[1]} features):\n')
        for model_name in ['tabpfn', 'lgbm', 'catboost', 'xgboost']:
            rmses, maes, r2s = [], [], []

            for train_idx, test_idx in kf.split(X):
                if model_name == 'tabpfn':
                    model = TabPFNRegressor.create_default_for_version(ModelVersion.V2_5, n_estimators=12, softmax_temperature=0.5, average_before_softmax=True)
                elif model_name == 'lgbm':
                    lgbm_kwargs = dict(
                        n_estimators=500,
                        learning_rate=0.05,
                        num_leaves=15,
                        min_data_in_leaf=5,
                        verbosity=-1
                    )
                    model = LGBMRegressor(**lgbm_kwargs)
                elif model_name == 'catboost':
                    catboost_kwargs = dict(
                        num_trees=600,
                        learning_rate=0.03,
                        max_depth=6,
                        l2_leaf_reg=1.0,
                        subsample=0.8,
                        rsm=0.7,
                        verbose=0
                    )
                    model = CatBoostRegressor(**catboost_kwargs)
                elif model_name == 'xgboost':
                    xgboost_kwargs = dict(
                        n_estimators=600,
                        learning_rate=0.03,
                        max_depth=6,
                        min_child_weight=5,
                        subsample=0.8,
                        colsample_bytree=0.7,
                        reg_alpha=0.0,
                        reg_lambda=1.0,
                    )
                    model = XGBRegressor(**xgboost_kwargs)

                X_train_fold = X[train_idx]
                X_test_fold  = X[test_idx]

                y_train_fold = y[train_idx]
                y_test_fold  = y[test_idx]
                model.fit(X=X_train_fold, y=y_train_fold)

                y_pred = model.predict(X_test_fold)

                rmses.append(np.sqrt(mean_squared_error(y_test_fold, y_pred)))
                maes.append(mean_absolute_error(y_test_fold, y_pred))
                r2s.append(r2_score(y_test_fold, y_pred))

            rmse_mean = np.mean(rmses)
            if best_rmse is None or rmse_mean < best_rmse:
                best_rmse = rmse_mean
                best_model = model_name
            results.append({
                'model_name': model_name, 'RMSE': round(np.mean(rmses), 4), 'MAE': round(np.mean(maes), 4), 'R2': round(np.mean(r2s), 4)
            })

        best_in_folder[best_model] += 1
        for res in results:
            print(res)
        print('\n')
    print(f'best in folder {folder_name}: {best_in_folder}')


ctopo_moldesc_by_topo_5_atom_pair Gd_ctopo_fp_ligand_moldesc_atom_pair_by_topo_5 (1579 features):

{'model_name': 'tabpfn', 'RMSE': 3.3675, 'MAE': 2.5698, 'R2': 0.5162}
{'model_name': 'lgbm', 'RMSE': 3.7871, 'MAE': 2.8373, 'R2': 0.3934}
{'model_name': 'catboost', 'RMSE': 3.5793, 'MAE': 2.7271, 'R2': 0.452}
{'model_name': 'xgboost', 'RMSE': 3.4263, 'MAE': 2.6139, 'R2': 0.4971}


ctopo_moldesc_by_topo_5_atom_pair Gd_ctopo_fp_skl_da_moldesc_atom_pair_by_topo_5 (678 features):

{'model_name': 'tabpfn', 'RMSE': 3.467, 'MAE': 2.6503, 'R2': 0.4815}
{'model_name': 'lgbm', 'RMSE': 3.7487, 'MAE': 2.8291, 'R2': 0.407}
{'model_name': 'catboost', 'RMSE': 3.6228, 'MAE': 2.779, 'R2': 0.438}
{'model_name': 'xgboost', 'RMSE': 3.4963, 'MAE': 2.6435, 'R2': 0.474}


ctopo_moldesc_by_topo_5_atom_pair Gd_ctopo_fp_skl_da_skl_bonds_moldesc_atom_pair_by_topo_5 (721 features):

{'model_name': 'tabpfn', 'RMSE': 3.6149, 'MAE': 2.7115, 'R2': 0.428}
{'model_name': 'lgbm', 'RMSE': 3.7698, 'MAE': 2.8603, 'R2': 0.3972

# Fedot launches

Results:<br>
Gd_ctopo_fp_cmplx_moldesc_atom_pair: tabpfn rmse - 2,8758, fedot rmse - 2.9616<br>
Gd_ctopo_fp_skl_moldesc_atom_pair: tabpfn rmse - 3,0777, fedot rmse - 3.0906<br>
Gd_ctopo_fp_topo_moldesc_maccs: tabpfn rmse - 2,7917, fedot rmse - 2.8174<br>
Gd_ctopo_fp_skl_da_moldesc_maccs: tabpfn rmse - 2,8799, fedot rmse - 2.9029<br>
Gd_ctopo_fp_skl_da_skl_moldesc_morgan: tabpfn rmse - 2,9497, fedot rmse - 2.9485<br>

FEDOT constructed pipelines examples can be found below, by searching "Pipeline structure:"

In [5]:
from fedot import Fedot
from fedot.core.pipelines.pipeline_builder import PipelineBuilder


available_operations = ['tabpfnreg', 'adareg', 'catboostreg', 'fast_ica', 'isolation_forest_reg', 'knnreg', 'lasso',
                        'lgbmreg', 'linear', 'normalization', 'pca', 'poly_features', 'rfr', 'ridge', 'scaling',
                        'xgboostreg']

for folder_name, datasets_dict in datasets_v3.items():
    for name, df in datasets_dict.items():
        if name not in [
            'Gd_ctopo_fp_cmplx_moldesc_atom_pair', 'Gd_ctopo_fp_skl_moldesc_atom_pair',
            'Gd_ctopo_fp_topo_moldesc_maccs', 'Gd_ctopo_fp_skl_da_moldesc_maccs',
            'Gd_ctopo_fp_skl_da_skl_moldesc_morgan'
        ]:
            continue

        feature_cols = [c for c in df.columns if c != "lgK"]
        X = df[feature_cols].astype(np.float64).values
        y = df["lgK"].values

        vt = VarianceThreshold(threshold=0)
        X = vt.fit_transform(X)
        if X.shape[1] > 2000:
            vt = VarianceThreshold(threshold=0.01)
            X = vt.fit_transform(X)

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(kf.split(X))
        results = []
        best_rmse, best_model = None, None

        rmses, maes, r2s = [], [], []

        for train_idx, test_idx in kf.split(X):
            initial_assumption = PipelineBuilder().add_node('tabpfnreg', params={'n_estimators': 12, 'softmax_temperature': 0.5, 'average_before_softmax': True, 'max_features': 2000}).build()
            model = Fedot(
                problem='regression',
                timeout=20,
                n_jobs=-1,
                logging_level=50,
                available_operations=available_operations,
                initial_assumption=initial_assumption
            )

            X_train_fold = X[train_idx]
            X_test_fold = X[test_idx]

            y_train_fold = y[train_idx]
            y_test_fold = y[test_idx]

            pipeline = model.fit(features=X_train_fold, target=y_train_fold)
            pipeline.print_structure()

            y_pred = model.predict(X_test_fold)

            print(rmses, maes, r2s)
            rmses.append(np.sqrt(mean_squared_error(y_test_fold, y_pred)))
            maes.append(mean_absolute_error(y_test_fold, y_pred))
            r2s.append(r2_score(y_test_fold, y_pred))

        print(f'{folder_name} {name} ({X.shape[1]}, features):\n')
        print(f"RMSE: {np.mean(rmses):.4f}, MAE: {np.mean(maes):.4f}, R2: {np.mean(r2s):.4f}")
        print('\n')


Generations:   0%|          | 0/10000 [00:00<?, ?gen/s]

2025-12-17 23:05:28,414 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-17 23:05:28,429 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_default.ckpt
2025-12-17 23:05:28,844 - HuggingFace download failed.
2025-12-17 23:05:28,845 - For commercial usage, we provide alternative download options for v2.5, please reach out to us at sales@priorlabs.ai.
2025-12-17 23:06:59,371 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-17 23:06:59,400 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_default.ckpt
2025-12-17 23:06:59,785 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-17 23:06:59,803 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_default.ckpt
2025-12-17 23:06:59,814 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-17 23:06:59,844 - Attempting HuggingFace download: tabpfn-v2.5-regressor

Generations:   0%|          | 1/10000 [03:35<597:22:57, 215.08s/gen]

2025-12-17 23:11:08,632 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-17 23:11:08,665 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_default.ckpt
2025-12-17 23:11:10,130 - HuggingFace download failed.
2025-12-17 23:11:10,130 - For commercial usage, we provide alternative download options for v2.5, please reach out to us at sales@priorlabs.ai.


Generations:   0%|          | 2/10000 [06:48<562:33:22, 202.56s/gen]

...

Generations:   0%|          | 3/10000 [05:04<268:45:54, 96.78s/gen] 

2025-12-18 06:16:55,741 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-18 06:16:55,746 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-18 06:16:55,767 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_default.ckpt
2025-12-18 06:16:55,775 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_default.ckpt
2025-12-18 06:16:56,144 - HuggingFace download failed.
2025-12-18 06:16:56,144 - For commercial usage, we provide alternative download options for v2.5, please reach out to us at sales@priorlabs.ai.
2025-12-18 06:16:56,150 - HuggingFace download failed.
2025-12-18 06:16:56,151 - For commercial usage, we provide alternative download options for v2.5, please reach out to us at sales@priorlabs.ai.
2025-12-18 06:16:59,388 - Downloading model to /tmp/FEDOT/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt.
2025-12-18 06:16:59,420 - Attempting HuggingFace download: tabpfn-v2.5-regressor-v2.5_

Generations:   0%|          | 12/10000 [09:39<133:56:34, 48.28s/gen]


  0%|          | 24/100000 [07:07<494:46:31, 17.82s/trial, best loss: 2.968705267572518] 
Pipeline structure:
{'depth': 4, 'length': 4, 'nodes': [lasso, isolation_forest_reg, fast_ica, tabpfnreg]}
lasso - {'alpha': 0.13634897500461962}
isolation_forest_reg - {'bootstrap': True, 'max_features': 0.2846942112201904, 'max_samples': 0.4636351933886087}
fast_ica - {'whiten': 'unit-variance', 'fun': 'logcosh', 'n_components': 12}
tabpfnreg - {'n_jobs': 20, 'n_estimators': 5, 'softmax_temperature': 0.5386919674879675, 'average_before_softmax': True, 'model_path': 'auto', 'device': 'cuda', 'ignore_pretraining_limits': False, 'inference_precision': 'auto', 'fit_mode': 'fit_preprocessors', 'memory_saving_mode': 'auto', 'inference_config': None, 'enable_categorical': True, 'max_samples': 1000, 'max_features': 2000}
[2.5185066954466078, 3.2744198463903493, 3.1443751967860307, 2.5330799724923163] [1.914444951144131, 2.3533911522951994, 2.301519963464072, 1.9154469738450164] [0.6730496593173891, 0.37

"\ndidn't launched it here, but the results worse than just tabpfn. Produced pipelines examples:\n{'depth': 3, 'length': 3, 'nodes': [lasso, pca, resample]}\n{'depth': 4, 'length': 4, 'nodes': [ridge, scaling, rfr, poly_features]}\n{'depth': 3, 'length': 3, 'nodes': [lgbmreg, poly_features, normalization]}\n{'depth': 5, 'length': 5, 'nodes': [ridge, scaling, poly_features, poly_features, resample]}\n"